# Novelty Detection Models for Bbox Filtering

This notebook implements and tests novelty detection models to identify anomalous bounding boxes.

**Models:**
- LocalOutlierFactor (with novelty=True)
- OneClassSVM
- IsolationForest

**Features:**
- bbox aspect ratio (w/h)
- bbox area (w * h)
- distance from keypoint position to frame center
- camera height

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import json

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load and Prepare Data

In [ ]:
# Load the CSV file
CSV_FILE = "yolo_annotations.csv"

df = pd.read_csv(CSV_FILE)
print(f"Loaded {len(df)} annotations")
df.head()

In [ ]:
# Calculate features
df['bbox_area'] = df['w'] * df['h']
df['aspect_ratio'] = df['w'] / df['h']

# Distance from center (normalized coordinates)
# Center is at (0.5, 0.5) in normalized YOLO format
df['dist_from_center_x'] = df['x'] - 0.5
df['dist_from_center_y'] = df['y'] - 0.5
df['dist_from_center'] = np.sqrt(df['dist_from_center_x']**2 + df['dist_from_center_y']**2)

print("\nFeatures calculated:")
print(f"- bbox_area: {df['bbox_area'].describe()}")
print(f"\n- aspect_ratio: {df['aspect_ratio'].describe()}")
print(f"\n- dist_from_center: {df['dist_from_center'].describe()}")

In [ ]:
# Prepare feature matrix
feature_cols = ['aspect_ratio', 'bbox_area', 'dist_from_center', 'cam_height']
X = df[feature_cols].values

print(f"Feature matrix shape: {X.shape}")
print(f"\nFeature statistics:")
print(pd.DataFrame(X, columns=feature_cols).describe())

In [ ]:
# Split data: use train subset for training, others for testing
train_mask = df['subset'] == 'train'
X_train = X[train_mask]
X_test = X[~train_mask]

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

In [ ]:
# Feature scaling (important for SVM and LOF)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed")
print(f"\nScaled training data shape: {X_train_scaled.shape}")
print(f"Mean: {X_train_scaled.mean(axis=0)}")
print(f"Std: {X_train_scaled.std(axis=0)}")

## 2. Coordinate Conversion Utilities

In [ ]:
def absolute_to_relative(x_abs, y_abs, w_abs, h_abs, img_width, img_height):
    """
    Convert absolute bbox coordinates to relative (normalized) YOLO format.
    
    Args:
        x_abs: absolute x coordinate (top-left corner)
        y_abs: absolute y coordinate (top-left corner)
        w_abs: absolute width
        h_abs: absolute height
        img_width: image width in pixels
        img_height: image height in pixels
    
    Returns:
        tuple: (x_center, y_center, w_rel, h_rel) in normalized coordinates [0, 1]
    """
    # Convert to relative coordinates
    x_rel = x_abs / img_width
    y_rel = y_abs / img_height
    w_rel = w_abs / img_width
    h_rel = h_abs / img_height
    
    # Convert from top-left to center coordinates
    x_center = x_rel + w_rel / 2
    y_center = y_rel + h_rel / 2
    
    return x_center, y_center, w_rel, h_rel


def relative_to_absolute(x_center, y_center, w_rel, h_rel, img_width, img_height):
    """
    Convert relative (normalized) YOLO format to absolute bbox coordinates.
    
    Args:
        x_center: normalized x center coordinate [0, 1]
        y_center: normalized y center coordinate [0, 1]
        w_rel: normalized width [0, 1]
        h_rel: normalized height [0, 1]
        img_width: image width in pixels
        img_height: image height in pixels
    
    Returns:
        tuple: (x_abs, y_abs, w_abs, h_abs) in absolute pixel coordinates
    """
    # Convert from center to top-left coordinates
    x_rel = x_center - w_rel / 2
    y_rel = y_center - h_rel / 2
    
    # Convert to absolute coordinates
    x_abs = int(x_rel * img_width)
    y_abs = int(y_rel * img_height)
    w_abs = int(w_rel * img_width)
    h_abs = int(h_rel * img_height)
    
    return x_abs, y_abs, w_abs, h_abs


# Test the conversion functions
print("Testing coordinate conversion:")
print("="*60)

# Example: 640x480 image
img_w, img_h = 640, 480

# Absolute coordinates: bbox at (100, 150) with size 200x300
x_abs, y_abs, w_abs, h_abs = 100, 150, 200, 300
print(f"\nAbsolute: x={x_abs}, y={y_abs}, w={w_abs}, h={h_abs}")

# Convert to relative
x_rel, y_rel, w_rel, h_rel = absolute_to_relative(x_abs, y_abs, w_abs, h_abs, img_w, img_h)
print(f"Relative: x={x_rel:.4f}, y={y_rel:.4f}, w={w_rel:.4f}, h={h_rel:.4f}")

# Convert back to absolute
x_abs2, y_abs2, w_abs2, h_abs2 = relative_to_absolute(x_rel, y_rel, w_rel, h_rel, img_w, img_h)
print(f"Back to absolute: x={x_abs2}, y={y_abs2}, w={w_abs2}, h={h_abs2}")

# Verify
assert (x_abs, y_abs, w_abs, h_abs) == (x_abs2, y_abs2, w_abs2, h_abs2), "Conversion failed!"
print("\n✓ Conversion test passed!")

In [ ]:
def extract_features_from_bbox(x_center, y_center, w_rel, h_rel, cam_height):
    """
    Extract features from a bbox for novelty detection.
    
    Args:
        x_center: normalized x center coordinate [0, 1]
        y_center: normalized y center coordinate [0, 1]
        w_rel: normalized width [0, 1]
        h_rel: normalized height [0, 1]
        cam_height: camera height in cm
    
    Returns:
        numpy array: feature vector [aspect_ratio, bbox_area, dist_from_center, cam_height]
    """
    aspect_ratio = w_rel / h_rel if h_rel > 0 else 0
    bbox_area = w_rel * h_rel
    dist_from_center = np.sqrt((x_center - 0.5)**2 + (y_center - 0.5)**2)
    
    return np.array([aspect_ratio, bbox_area, dist_from_center, cam_height])


# Test feature extraction
print("Testing feature extraction:")
print("="*60)

# Use the relative coordinates from above
features = extract_features_from_bbox(x_rel, y_rel, w_rel, h_rel, cam_height=250)
print(f"\nFeatures: {features}")
print(f"  Aspect ratio: {features[0]:.4f}")
print(f"  Bbox area: {features[1]:.4f}")
print(f"  Distance from center: {features[2]:.4f}")
print(f"  Camera height: {features[3]:.0f} cm")

## 3. Train Novelty Detection Models

### 3.1 Local Outlier Factor (LOF)

In [ ]:
# Train LocalOutlierFactor with novelty=True
lof_model = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.1,  # Expected proportion of outliers
    novelty=True  # Enable novelty detection
)

print("Training LocalOutlierFactor...")
lof_model.fit(X_train_scaled)
print("✓ LOF model trained")

# Test on training data
lof_train_pred = lof_model.predict(X_train_scaled)
lof_train_scores = lof_model.score_samples(X_train_scaled)

print(f"\nTraining set:")
print(f"  Inliers: {(lof_train_pred == 1).sum()} ({(lof_train_pred == 1).sum()/len(lof_train_pred)*100:.2f}%)")
print(f"  Outliers: {(lof_train_pred == -1).sum()} ({(lof_train_pred == -1).sum()/len(lof_train_pred)*100:.2f}%)")
print(f"  Score range: [{lof_train_scores.min():.3f}, {lof_train_scores.max():.3f}]")

# Test on test data
lof_test_pred = lof_model.predict(X_test_scaled)
lof_test_scores = lof_model.score_samples(X_test_scaled)

print(f"\nTest set:")
print(f"  Inliers: {(lof_test_pred == 1).sum()} ({(lof_test_pred == 1).sum()/len(lof_test_pred)*100:.2f}%)")
print(f"  Outliers: {(lof_test_pred == -1).sum()} ({(lof_test_pred == -1).sum()/len(lof_test_pred)*100:.2f}%)")
print(f"  Score range: [{lof_test_scores.min():.3f}, {lof_test_scores.max():.3f}]")

### 3.2 One-Class SVM

In [ ]:
# Train OneClassSVM
svm_model = OneClassSVM(
    kernel='rbf',
    gamma='auto',
    nu=0.1  # Upper bound on fraction of outliers
)

print("Training OneClassSVM...")
svm_model.fit(X_train_scaled)
print("✓ SVM model trained")

# Test on training data
svm_train_pred = svm_model.predict(X_train_scaled)
svm_train_scores = svm_model.score_samples(X_train_scaled)

print(f"\nTraining set:")
print(f"  Inliers: {(svm_train_pred == 1).sum()} ({(svm_train_pred == 1).sum()/len(svm_train_pred)*100:.2f}%)")
print(f"  Outliers: {(svm_train_pred == -1).sum()} ({(svm_train_pred == -1).sum()/len(svm_train_pred)*100:.2f}%)")
print(f"  Score range: [{svm_train_scores.min():.3f}, {svm_train_scores.max():.3f}]")

# Test on test data
svm_test_pred = svm_model.predict(X_test_scaled)
svm_test_scores = svm_model.score_samples(X_test_scaled)

print(f"\nTest set:")
print(f"  Inliers: {(svm_test_pred == 1).sum()} ({(svm_test_pred == 1).sum()/len(svm_test_pred)*100:.2f}%)")
print(f"  Outliers: {(svm_test_pred == -1).sum()} ({(svm_test_pred == -1).sum()/len(svm_test_pred)*100:.2f}%)")
print(f"  Score range: [{svm_test_scores.min():.3f}, {svm_test_scores.max():.3f}]")

### 3.3 Isolation Forest

In [ ]:
# Train IsolationForest
iforest_model = IsolationForest(
    n_estimators=30,
    # contamination=0.1,  # Expected proportion of outliers
    random_state=42
)

print("Training IsolationForest...")
iforest_model.fit(X_train_scaled)
print("✓ IsolationForest model trained")

# Test on training data
iforest_train_pred = iforest_model.predict(X_train_scaled)
iforest_train_scores = iforest_model.score_samples(X_train_scaled)

print(f"\nTraining set:")
print(f"  Inliers: {(iforest_train_pred == 1).sum()} ({(iforest_train_pred == 1).sum()/len(iforest_train_pred)*100:.2f}%)")
print(f"  Outliers: {(iforest_train_pred == -1).sum()} ({(iforest_train_pred == -1).sum()/len(iforest_train_pred)*100:.2f}%)")
print(f"  Score range: [{iforest_train_scores.min():.3f}, {iforest_train_scores.max():.3f}]")

# Test on test data
iforest_test_pred = iforest_model.predict(X_test_scaled)
iforest_test_scores = iforest_model.score_samples(X_test_scaled)

print(f"\nTest set:")
print(f"  Inliers: {(iforest_test_pred == 1).sum()} ({(iforest_test_pred == 1).sum()/len(iforest_test_pred)*100:.2f}%)")
print(f"  Outliers: {(iforest_test_pred == -1).sum()} ({(iforest_test_pred == -1).sum()/len(iforest_test_pred)*100:.2f}%)")
print(f"  Score range: [{iforest_test_scores.min():.3f}, {iforest_test_scores.max():.3f}]")

## 4. Compare Models

In [ ]:
# Visualize score distributions
fig, axes = plt.subplots(3, 2, figsize=(16, 12))

models = [
    ('LOF', lof_train_scores, lof_test_scores),
    ('One-Class SVM', svm_train_scores, svm_test_scores),
    ('Isolation Forest', iforest_train_scores, iforest_test_scores)
]

for idx, (model_name, train_scores, test_scores) in enumerate(models):
    # Training scores
    axes[idx, 0].hist(train_scores, bins=50, alpha=0.7, color='blue', edgecolor='black')
    axes[idx, 0].axvline(0, color='red', linestyle='--', linewidth=2, label='Decision boundary')
    axes[idx, 0].set_xlabel('Anomaly Score', fontsize=12)
    axes[idx, 0].set_ylabel('Frequency', fontsize=12)
    axes[idx, 0].set_title(f'{model_name} - Training Set', fontsize=14, fontweight='bold')
    axes[idx, 0].legend()
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Test scores
    axes[idx, 1].hist(test_scores, bins=50, alpha=0.7, color='green', edgecolor='black')
    axes[idx, 1].axvline(0, color='red', linestyle='--', linewidth=2, label='Decision boundary')
    axes[idx, 1].set_xlabel('Anomaly Score', fontsize=12)
    axes[idx, 1].set_ylabel('Frequency', fontsize=12)
    axes[idx, 1].set_title(f'{model_name} - Test Set', fontsize=14, fontweight='bold')
    axes[idx, 1].legend()
    axes[idx, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Compare outlier detection rates
comparison_data = {
    'Model': ['LOF', 'One-Class SVM', 'Isolation Forest'],
    'Train Outliers (%)': [
        (lof_train_pred == -1).sum() / len(lof_train_pred) * 100,
        (svm_train_pred == -1).sum() / len(svm_train_pred) * 100,
        (iforest_train_pred == -1).sum() / len(iforest_train_pred) * 100
    ],
    'Test Outliers (%)': [
        (lof_test_pred == -1).sum() / len(lof_test_pred) * 100,
        (svm_test_pred == -1).sum() / len(svm_test_pred) * 100,
        (iforest_test_pred == -1).sum() / len(iforest_test_pred) * 100
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Comparison:")
print("="*60)
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df['Train Outliers (%)'], width, label='Training', alpha=0.8)
bars2 = ax.bar(x + width/2, comparison_df['Test Outliers (%)'], width, label='Test', alpha=0.8)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Outlier Detection Rate (%)', fontsize=12)
ax.set_title('Outlier Detection Rate Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Model'])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Test with Example Bboxes

In [ ]:
# Create test examples with different characteristics
test_examples = [
    # Normal examples (similar to training data)
    {'name': 'Normal bbox 1', 'x': 0.5, 'y': 0.5, 'w': 0.15, 'h': 0.2, 'cam_height': 250},
    {'name': 'Normal bbox 2', 'x': 0.3, 'y': 0.6, 'w': 0.12, 'h': 0.18, 'cam_height': 260},
    
    # Anomalous examples
    {'name': 'Very large bbox', 'x': 0.5, 'y': 0.5, 'w': 0.8, 'h': 0.9, 'cam_height': 250},
    {'name': 'Very small bbox', 'x': 0.5, 'y': 0.5, 'w': 0.01, 'h': 0.01, 'cam_height': 250},
    {'name': 'Extreme aspect ratio (wide)', 'x': 0.5, 'y': 0.5, 'w': 0.6, 'h': 0.05, 'cam_height': 250},
    {'name': 'Extreme aspect ratio (tall)', 'x': 0.5, 'y': 0.5, 'w': 0.05, 'h': 0.6, 'cam_height': 250},
    {'name': 'Unusual camera height', 'x': 0.5, 'y': 0.5, 'w': 0.15, 'h': 0.2, 'cam_height': 500},
    {'name': 'Far from center', 'x': 0.95, 'y': 0.95, 'w': 0.15, 'h': 0.2, 'cam_height': 250},
]

print("Testing with example bboxes:")
print("="*80)

results = []

for example in test_examples:
    # Extract features
    features = extract_features_from_bbox(
        example['x'], example['y'], example['w'], example['h'], example['cam_height']
    )
    
    # Scale features
    features_scaled = scaler.transform(features.reshape(1, -1))
    
    # Predict with all models
    lof_pred = lof_model.predict(features_scaled)[0]
    lof_score = lof_model.score_samples(features_scaled)[0]
    
    svm_pred = svm_model.predict(features_scaled)[0]
    svm_score = svm_model.score_samples(features_scaled)[0]
    
    iforest_pred = iforest_model.predict(features_scaled)[0]
    iforest_score = iforest_model.score_samples(features_scaled)[0]
    
    results.append({
        'Name': example['name'],
        'Aspect Ratio': f"{features[0]:.3f}",
        'Area': f"{features[1]:.4f}",
        'Dist from Center': f"{features[2]:.3f}",
        'LOF': 'Normal' if lof_pred == 1 else 'Outlier',
        'LOF Score': f"{lof_score:.3f}",
        'SVM': 'Normal' if svm_pred == 1 else 'Outlier',
        'SVM Score': f"{svm_score:.3f}",
        'IForest': 'Normal' if iforest_pred == 1 else 'Outlier',
        'IForest Score': f"{iforest_score:.3f}"
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Save results
results_df.to_csv('example_predictions.csv', index=False)
print("\n✓ Results saved to example_predictions.csv")

## 6. Save Models

In [ ]:
# Save all models and scaler
models_to_save = {
    'lof': lof_model,
    'svm': svm_model,
    'iforest': iforest_model,
    'scaler': scaler
}

for name, model in models_to_save.items():
    filename = f'novelty_detection_{name}.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(model, f)
    print(f"✓ Saved {filename}")

print("\nAll models saved successfully!")

## 7. Usage Example: Load and Predict

In [ ]:
# Example: Load a model and make predictions
print("Example: Loading model and making predictions")
print("="*60)

# Load saved model and scaler
with open('novelty_detection_iforest.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

with open('novelty_detection_scaler.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)

print("✓ Models loaded\n")

# Example: Convert absolute coordinates and predict
print("Example prediction pipeline:")
print("-"*60)

# Image size
img_width, img_height = 640, 480

# Absolute bbox coordinates
x_abs, y_abs, w_abs, h_abs = 200, 150, 80, 120
cam_height = 250

print(f"Input (absolute coords):")
print(f"  Image size: {img_width}x{img_height}")
print(f"  Bbox: x={x_abs}, y={y_abs}, w={w_abs}, h={h_abs}")
print(f"  Camera height: {cam_height} cm")

# Step 1: Convert to relative coordinates
x_rel, y_rel, w_rel, h_rel = absolute_to_relative(x_abs, y_abs, w_abs, h_abs, img_width, img_height)
print(f"\nRelative coords: x={x_rel:.4f}, y={y_rel:.4f}, w={w_rel:.4f}, h={h_rel:.4f}")

# Step 2: Extract features
features = extract_features_from_bbox(x_rel, y_rel, w_rel, h_rel, cam_height)
print(f"\nFeatures: {features}")

# Step 3: Scale features
features_scaled = loaded_scaler.transform(features.reshape(1, -1))
print(f"Scaled features: {features_scaled[0]}")

# Step 4: Predict
prediction = loaded_model.predict(features_scaled)[0]
score = loaded_model.score_samples(features_scaled)[0]

print(f"\n{'='*60}")
print(f"Prediction: {'NORMAL' if prediction == 1 else 'OUTLIER'}")
print(f"Anomaly score: {score:.4f}")
print(f"{'='*60}")

## 8. Complete Pipeline: Predict from Absolute Coordinates

In [ ]:
def predict_bbox_novelty(x, y, w, h, cam_height, img_width, img_height, 
                         lof_model, svm_model, iforest_model, scaler, verbose=True):
    """
    Complete pipeline: predict if a bbox is novel/anomalous using all three models.
    
    Args:
        x: absolute x coordinate (top-left corner) in pixels
        y: absolute y coordinate (top-left corner) in pixels
        w: absolute width in pixels
        h: absolute height in pixels
        cam_height: camera height in cm
        img_width: image width in pixels
        img_height: image height in pixels
        lof_model: trained LocalOutlierFactor model
        svm_model: trained OneClassSVM model
        iforest_model: trained IsolationForest model
        scaler: fitted StandardScaler
        verbose: if True, print detailed results
    
    Returns:
        dict: predictions and scores from all three models
    """
    if verbose:
        print("="*80)
        print("BBOX NOVELTY DETECTION PIPELINE")
        print("="*80)
        print(f"\nInput (absolute coordinates):")
        print(f"  Image size: {img_width}x{img_height}")
        print(f"  Bbox top-left: ({x}, {y})")
        print(f"  Bbox size: {w}x{h}")
        print(f"  Camera height: {cam_height} cm")
    
    # Step 1: Calculate center point
    x_center_abs = x + w / 2
    y_center_abs = y + h / 2
    
    if verbose:
        print(f"\nStep 1: Calculate center point")
        print(f"  Center (absolute): ({x_center_abs:.1f}, {y_center_abs:.1f})")
    
    # Step 2: Convert to relative coordinates
    x_center_rel, y_center_rel, w_rel, h_rel = absolute_to_relative(x, y, w, h, img_width, img_height)
    
    if verbose:
        print(f"\nStep 2: Convert to relative coordinates")
        print(f"  Center (relative): ({x_center_rel:.4f}, {y_center_rel:.4f})")
        print(f"  Size (relative): w={w_rel:.4f}, h={h_rel:.4f}")
    
    # Step 3: Extract features
    features = extract_features_from_bbox(x_center_rel, y_center_rel, w_rel, h_rel, cam_height)
    
    if verbose:
        print(f"\nStep 3: Extract features")
        print(f"  Aspect ratio (w/h): {features[0]:.4f}")
        print(f"  Bbox area: {features[1]:.4f}")
        print(f"  Distance from center: {features[2]:.4f}")
        print(f"  Camera height: {features[3]:.0f} cm")
    
    # Step 4: Scale features
    features_scaled = scaler.transform(features.reshape(1, -1))
    
    if verbose:
        print(f"\nStep 4: Scale features")
        print(f"  Scaled features: [{', '.join([f'{f:.4f}' for f in features_scaled[0]])}]")
    
    # Step 5: Run all three models
    lof_pred = lof_model.predict(features_scaled)[0]
    lof_score = lof_model.score_samples(features_scaled)[0]
    
    svm_pred = svm_model.predict(features_scaled)[0]
    svm_score = svm_model.score_samples(features_scaled)[0]
    
    iforest_pred = iforest_model.predict(features_scaled)[0]
    iforest_score = iforest_model.score_samples(features_scaled)[0]
    
    if verbose:
        print(f"\n{'='*80}")
        print("RESULTS")
        print("="*80)
        print(f"\n{'Model':<20} {'Prediction':<12} {'Score':<15} {'Status'}")
        print("-"*80)
        print(f"{'LocalOutlierFactor':<20} {'Normal' if lof_pred == 1 else 'Outlier':<12} {lof_score:<15.4f} {'✓' if lof_pred == 1 else '✗'}")
        print(f"{'One-Class SVM':<20} {'Normal' if svm_pred == 1 else 'Outlier':<12} {svm_score:<15.4f} {'✓' if svm_pred == 1 else '✗'}")
        print(f"{'Isolation Forest':<20} {'Normal' if iforest_pred == 1 else 'Outlier':<12} {iforest_score:<15.4f} {'✓' if iforest_pred == 1 else '✗'}")
        print("-"*80)
        
        # Consensus
        outlier_count = sum([lof_pred == -1, svm_pred == -1, iforest_pred == -1])
        print(f"\nConsensus: {outlier_count}/3 models flagged as outlier")
        if outlier_count == 0:
            print("  → ACCEPT (all models agree: NORMAL)")
        elif outlier_count == 3:
            print("  → REJECT (all models agree: OUTLIER)")
        else:
            print("  → WARNING (models disagree - manual review recommended)")
        print("="*80)
    
    return {
        'lof_prediction': 'normal' if lof_pred == 1 else 'outlier',
        'lof_score': lof_score,
        'svm_prediction': 'normal' if svm_pred == 1 else 'outlier',
        'svm_score': svm_score,
        'iforest_prediction': 'normal' if iforest_pred == 1 else 'outlier',
        'iforest_score': iforest_score,
        'outlier_count': sum([lof_pred == -1, svm_pred == -1, iforest_pred == -1]),
        'features': {
            'aspect_ratio': features[0],
            'bbox_area': features[1],
            'dist_from_center': features[2],
            'cam_height': features[3]
        }
    }


print("✓ Function defined: predict_bbox_novelty()")

### Test Examples

In [ ]:
# Test 1: Normal bbox
print("TEST 1: Normal bbox")
result1 = predict_bbox_novelty(
    x=200, y=150, w=80, h=120,
    cam_height=250,
    img_width=640, img_height=480,
    lof_model=lof_model,
    svm_model=svm_model,
    iforest_model=iforest_model,
    scaler=scaler
)
print(f"\nReturned dict: {result1}")
print("\n" + "="*80 + "\n")

In [ ]:
# Test 2: Very large bbox (anomalous)
print("TEST 2: Very large bbox (anomalous)")
result2 = predict_bbox_novelty(
    x=50, y=50, w=500, h=400,
    cam_height=260,
    img_width=640, img_height=480,
    lof_model=lof_model,
    svm_model=svm_model,
    iforest_model=iforest_model,
    scaler=scaler
)
print(f"\nReturned dict: {result2}")
print("\n" + "="*80 + "\n")

In [ ]:
# Test 3: Extreme aspect ratio (wide bbox)
print("TEST 3: Extreme aspect ratio - very wide bbox")
result3 = predict_bbox_novelty(
    x=100, y=200, w=400, h=30,
    cam_height=250,
    img_width=640, img_height=480,
    lof_model=lof_model,
    svm_model=svm_model,
    iforest_model=iforest_model,
    scaler=scaler
)
print(f"\nReturned dict: {result3}")
print("\n" + "="*80 + "\n")

In [ ]:
# Test 4: Bbox at unusual camera height
print("TEST 4: Unusual camera height")
result4 = predict_bbox_novelty(
    x=220, y=180, w=75, h=110,
    cam_height=450,  # Much higher than training data
    img_width=640, img_height=480,
    lof_model=lof_model,
    svm_model=svm_model,
    iforest_model=iforest_model,
    scaler=scaler
)
print(f"\nReturned dict: {result4}")
print("\n" + "="*80 + "\n")

In [ ]:
# Test 5: Multiple bboxes in a batch
print("TEST 5: Batch processing multiple bboxes")
print("="*80)

test_bboxes = [
    {'name': 'Small normal', 'x': 150, 'y': 100, 'w': 60, 'h': 90, 'cam_height': 250},
    {'name': 'Large normal', 'x': 180, 'y': 120, 'w': 100, 'h': 140, 'cam_height': 260},
    {'name': 'Tiny bbox', 'x': 300, 'y': 200, 'w': 5, 'h': 8, 'cam_height': 250},
    {'name': 'Off-center', 'x': 550, 'y': 420, 'w': 70, 'h': 100, 'cam_height': 270},
]

batch_results = []
for i, bbox in enumerate(test_bboxes, 1):
    print(f"\n[{i}/{len(test_bboxes)}] Processing: {bbox['name']}")
    print("-"*80)
    result = predict_bbox_novelty(
        x=bbox['x'], y=bbox['y'], w=bbox['w'], h=bbox['h'],
        cam_height=bbox['cam_height'],
        img_width=640, img_height=480,
        lof_model=lof_model,
        svm_model=svm_model,
        iforest_model=iforest_model,
        scaler=scaler,
        verbose=False  # Disable verbose for batch processing
    )
    batch_results.append({
        'name': bbox['name'],
        'outlier_count': result['outlier_count'],
        'lof': result['lof_prediction'],
        'svm': result['svm_prediction'],
        'iforest': result['iforest_prediction']
    })

# Summary table
print("\n" + "="*80)
print("BATCH PROCESSING SUMMARY")
print("="*80)
summary_df = pd.DataFrame(batch_results)
print(summary_df.to_string(index=False))
print("="*80)

In [ ]:
# Test 1: Normal bbox
# box: x = 63, y = 8, w = 81, h = 81
print("TEST 1: Normal bbox")
result1 = predict_bbox_novelty(
    x=63, y=8, w=81, h=81,
    cam_height=310,
    img_width=640, img_height=480,
    lof_model=lof_model,
    svm_model=svm_model,
    iforest_model=iforest_model,
    scaler=scaler
)
print(f"\nReturned dict: {result1}")
print("\n" + "="*80 + "\n")

In [ ]:
def export_scaler(scaler):
    return {
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
        "var": scaler.var_.tolist(),
        "n_features": len(scaler.mean_),
    }

scaler_params = export_scaler(scaler)
scaler_params

In [ ]:
def export_tree(tree):
    t = tree.tree_
    return {
        "feature": t.feature.tolist(),        # int feature index per node
        "threshold": t.threshold.tolist(),    # split threshold
        "left": t.children_left.tolist(),     # index of left child
        "right": t.children_right.tolist(),   # index of right child
        "n_node_samples": t.n_node_samples.tolist(),  # samples at node
        "node_count": t.node_count,
        "max_depth": t.max_depth,
    }


def export_isolation_forest(model):
    return {
        "n_estimators": len(model.estimators_),
        "max_samples": model.max_samples,
        "contamination": model.contamination,
        "trees": [export_tree(estimator) for estimator in model.estimators_],
    }

In [ ]:
iforest_params = export_isolation_forest(iforest_model)
iforest_params["n_estimators"]

In [ ]:
# Save parameters to JSON files
with open('scaler_params.json', 'w') as f:
    json.dump(scaler_params, f, indent=2)
print("✓ Saved scaler_params.json")

with open('iforest_params.json', 'w') as f:
    json.dump(iforest_params, f, indent=2)
print("✓ Saved iforest_params.json")

print("\nJSON export completed successfully!")